# vPCF Data Preprocessing and DEC/IDEC Training Pipeline

This notebook demonstrates how to:
1. Load vPCF data from HDF5 (.h5) and DM3 (.dm3) files
2. Preprocess and extract features
3. Train DEC and IDEC clustering models

## Data Files:
- `vPCF_test_2.h5` - HDF5 file with 7205 frames of 999x999 vPCF images
- `vPCF_test_2.dm3` - Digital Micrograph file with 1680x1680 image

In [ ]:
# Setup imports
import sys
import os
import numpy as np

# Add paths
sys.path.insert(0, '.')  # Current folder
sys.path.insert(0, '..')  # Parent for src imports

from vpcf_data_loader import (
    load_vpcf_file,
    combine_datasets,
    check_dependencies,
    print_file_info,
    VPCFDataset
)

# Check available dependencies
print("Available dependencies:")
for name, available in check_dependencies().items():
    status = "YES" if available else "NO"
    print(f"  {name}: {status}")

## 1. Define Data File Paths

In [ ]:
# Data file paths
H5_FILE = r"Experimentally-obtained vPCF Testing\data\vPCF_test_2.h5"
DM3_FILE = r"Experimentally-obtained vPCF Testing\data\vPCF_test_2.dm3"

# Verify files exist
print("File availability:")
print(f"  H5 file exists: {os.path.exists(H5_FILE)}")
print(f"  DM3 file exists: {os.path.exists(DM3_FILE)}")

## 2. Inspect Data File Structures

In [ ]:
# Inspect HDF5 file structure
print("=" * 50)
print("HDF5 File Structure:")
print("=" * 50)
print_file_info(H5_FILE)

In [ ]:
# Inspect DM3 file
print("=" * 50)
print("DM3 File Structure:")
print("=" * 50)
print_file_info(DM3_FILE)

## 3. Load and Preprocess Data

### Feature Extraction Methods:
- `flatten`: Flatten images to 1D vectors (large feature dimension)
- `histogram`: Extract histogram features (64 bins, compact)
- `statistical`: Extract statistical features (7 features: mean, std, min, max, median, skewness, kurtosis)
- `combined`: Combine histogram and statistical features (71 features)

### Normalization Methods:
- `minmax`: Scale to [0, 1] range
- `standard`: Z-score normalization
- `l2`: L2 normalization

In [ ]:
# Load HDF5 data with histogram features
# Use max_frames to limit data size for faster training/testing

h5_dataset = load_vpcf_file(
    H5_FILE,
    feature_method="histogram",  # Options: "flatten", "histogram", "statistical", "combined"
    normalize="minmax",          # Options: "minmax", "standard", "l2"
    downsample_factor=None,       # Optional: downsample images before feature extraction
    max_frames=1000,              # Limit frames for faster testing (remove for full data)
    verbose=True
)

print(f"\nLoaded dataset: {h5_dataset}")

In [ ]:
# Load DM3 data
dm3_dataset = load_vpcf_file(
    DM3_FILE,
    feature_method="histogram",
    normalize="minmax",
    verbose=True
)

print(f"\nLoaded dataset: {dm3_dataset}")

In [ ]:
# Get feature matrix for training (using H5 data since it has more samples)
x = h5_dataset.features

print(f"Feature matrix shape: {x.shape}")
print(f"Feature statistics:")
print(f"  Min: {x.min():.4f}")
print(f"  Max: {x.max():.4f}")
print(f"  Mean: {x.mean():.4f}")
print(f"  Std: {x.std():.4f}")

## 4. Train DEC Model

In [ ]:
from src.DEC import DEC

# Model configuration
n_clusters = 10  # Number of clusters to find
hidden_dims = [500, 500, 2000]  # Autoencoder hidden layers
dims = [x.shape[1]] + hidden_dims + [n_clusters]

print(f"Network architecture: {dims}")
print(f"Input dimension: {x.shape[1]}")
print(f"Number of clusters: {n_clusters}")

# Create output directory
save_dir = './results/dec'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# Initialize DEC model
dec = DEC(dims=dims, n_clusters=n_clusters, save_dir=save_dir)

# Pretrain the autoencoder
print("Pretraining autoencoder...")
dec.pretrain(x, epochs=50, batch_size=256)

In [ ]:
# Train clustering layer
print("Training clustering layer...")
dec.compile(optimizer='sgd')

# Reduce maxiter for faster testing
dec_labels = dec.fit(
    x,
    y=None,  # No ground truth labels
    maxiter=2000,  # Reduce for testing, use 8000+ for full training
    update_interval=140,
    batch_size=256
)

print(f"\nDEC training complete!")
print(f"Cluster distribution: {np.bincount(dec_labels)}")

## 5. Train IDEC Model

In [ ]:
from src.IDEC import IDEC

# Create output directory
save_dir_idec = './results/idec'
os.makedirs(save_dir_idec, exist_ok=True)

# Initialize IDEC model
idec = IDEC(
    dims=dims,
    n_clusters=n_clusters,
    gamma=0.1,  # Weight for reconstruction loss
    save_dir=save_dir_idec
)

# Pretrain the autoencoder
print("Pretraining autoencoder...")
idec.pretrain(x, epochs=50, batch_size=256)

In [ ]:
# Train clustering layer
print("Training clustering layer...")
idec.compile(optimizer='sgd')

idec_labels = idec.fit(
    x,
    y=None,
    maxiter=2000,  # Reduce for testing
    update_interval=140,
    batch_size=256
)

print(f"\nIDEC training complete!")
print(f"Cluster distribution: {np.bincount(idec_labels)}")

## 6. Compare Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# DEC cluster distribution
axes[0].bar(range(n_clusters), np.bincount(dec_labels, minlength=n_clusters))
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Count')
axes[0].set_title('DEC Cluster Distribution')

# IDEC cluster distribution
axes[1].bar(range(n_clusters), np.bincount(idec_labels, minlength=n_clusters))
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Count')
axes[1].set_title('IDEC Cluster Distribution')

plt.tight_layout()
plt.savefig('./results/cluster_comparison.png', dpi=150)
plt.show()

## 7. Save Results

In [ ]:
import pandas as pd

# Save cluster assignments
results_df = pd.DataFrame({
    'sample_idx': np.arange(len(dec_labels)),
    'dec_cluster': dec_labels,
    'idec_cluster': idec_labels
})

results_df.to_csv('./results/cluster_assignments.csv', index=False)
print("Results saved to: results/cluster_assignments.csv")

print("\nCluster agreement between DEC and IDEC:")
agreement = (dec_labels == idec_labels).mean() * 100
print(f"  {agreement:.2f}% samples assigned to same cluster")

## Alternative: Use the Command-Line Pipeline

You can also run training from the command line:

```bash
cd "Experimentally-obtained vPCF Testing"

# Train on H5 file only
python train_vpcf_models.py --h5-file "C:\Users\alexg\Downloads\vPCF_test_2.h5" --model both --n-clusters 10

# Train on DM3 file only
python train_vpcf_models.py --dm3-file "C:\Users\alexg\Downloads\vPCF_test_2.dm3" --model both --n-clusters 5

# Use different feature extraction
python train_vpcf_models.py --h5-file "path/to/file.h5" --feature-method combined --model idec
```